In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from scipy.spatial.distance import pdist


1. EDA and Data Preparation

In [ ]:
path = Path("Data") / "Dataset1_BankClients.xlsx" # general path

data = pd.read_excel(path)

In [ ]:
# Check if there are missing values
data.isna().sum()

# Data description
data.describe()

In [ ]:
data.info()
print()

In [ ]:
print(f'Numero duplicati: {data.duplicated().sum()}')
print('Valori mancanti per colonna:\n', data.isna().sum())

In [ ]:
# Specify categorical variables
categorical_columns = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']

numerical_columns = ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']

# Split variables
numerical_features = data.drop(columns=categorical_columns)  # Exclude categorical variables
categorical_features = data[categorical_columns]  # Select categorical variables

# Convert categorical in typ 'category' (for OneHotEncoder)
categorical_features = categorical_features.astype('category')

# Normalize numerical variables
mm_scaler = MinMaxScaler() # min-max scaler
z_scaler = StandardScaler() # z scaler
X_num = mm_scaler.fit_transform(numerical_features) # numerical variables mm scaled
X_num_z = z_scaler.fit_transform(numerical_features) # numerical variables z scaled

# One-hot encoding categorical variables
encoder = OneHotEncoder(drop='first', handle_unknown='ignore')  # Dummy encoding - ignoring unknwown values
X_cat = encoder.fit_transform(categorical_features).toarray()  # Convert into a dense matrix

# Concatenation of numerical and categorical variables
X = np.hstack((X_num, X_cat))
X_z = np.hstack((X_num_z, X_cat))

Y = np.hstack((X_num, categorical_features)) # needed for kprototype distance, cause it doesn't want encoding


In [ ]:
for col in categorical_columns:
    counts = data[col].value_counts()

    plt.figure()
    counts.plot(kind="bar")

    plt.title(f"Distribuzione di {col}")
    plt.xlabel(col)
    plt.ylabel("Frequenza")

    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
for col in numerical_columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data[col].dropna(), bins=30, kde=True)
    plt.title(f"Distribuzione di {col}")
    plt.xlabel(col)
    plt.ylabel("Frequenza")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in numerical_columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=data[col])
    plt.title(col)
    plt.show()

In [ ]:
sns.pairplot(data[numerical_columns])

In [ ]:
for col in categorical_columns:
    print(data[col].value_counts(normalize=True).head())

In [ ]:
distances = pdist(data[numerical_columns], metric="euclidean")

sns.histplot(distances, bins=50)
plt.title("Distribuzione delle distanze")

2. Distances


2.1. K-Prototypes

$d(x,y) = \sum (x_i-y_i)^2 + \gamma \sum \delta(x_j,y_j)$,\
where $\delta=1$ if $x_j {=}\mathllap{/} y_j$ and $\delta=0$ if $x_j = y_j$

numericals -> distanza euclidea \
categoricals -> matching

In [ ]:
from kmodes.kprototypes import KPrototypes

kproto = KPrototypes(
    n_clusters=3, # k=3 gives the best silhouette score
    init="Cao",
    random_state=42
)

cat_indices = list(range(len(numerical_columns),
                         len(numerical_columns) + len(categorical_columns)))

clusters = kproto.fit_predict(
    Y,
    categorical=cat_indices
)

In [ ]:
data["cluster"] = clusters
data.groupby("cluster")[numerical_columns].mean()

In [ ]:
for col in categorical_columns:
    print(data.groupby("cluster")[col].value_counts(normalize=True))

In [ ]:
def kproto_distance(x_num, x_cat, y_num, y_cat, gamma=1.0):
    num_dist = np.sum((x_num - y_num) ** 2)
    cat_dist = np.sum(x_cat != y_cat)
    return num_dist + gamma * cat_dist

def compute_kproto_distance_matrix(X_num, X_cat, gamma=1.0):
    n = X_num.shape[0]
    D = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):
            d = kproto_distance(X_num[i], X_cat[i], X_num[j], X_cat[j], gamma)
            D[i, j] = d
            D[j, i] = d

    return D

from sklearn.metrics import silhouette_score

D = compute_kproto_distance_matrix(X_num, X_cat, gamma=1.0)

score = silhouette_score(D, clusters, metric="precomputed")
print("Silhouette score:", score)

In [ ]:
from sklearn.metrics import davies_bouldin_score

db_score = davies_bouldin_score(X,clusters)
print(db_score)

2.2. Personalized Mixed Distance

$d(x,y) = \alpha d_{num} + (1-\alpha)d_{cat}$